# 02 - SILVER: Clean + Feature Engineering (Delta)

**Goal:** Transform raw Bronze data into analytics-ready Silver tables:
1) `silver.appointments_clean` — clean, typed, enriched appointment records  
2) `silver.patient_history_features` — patient history features (no leakage)  
3) `silver.model_features` — model-ready feature table (joined)

**DQ Gates:** Log Silver checks into `dq.check_results`.


In [0]:
%sql
CREATE DATABASE IF NOT EXISTS silver;

## 1) Create `silver.appointments_clean`

Cleans and standardizes types, timestamps, labels, and creates derived features:
- `lead_time_days`, `is_weekend`, `is_senior`
- filters invalid ages (0–110)


In [0]:
%sql
CREATE OR REPLACE TABLE silver.appointments_clean
USING DELTA
AS
SELECT
  CAST(AppointmentID AS STRING) AS appointment_id,
  CAST(PatientId AS STRING)     AS patient_id,

  to_timestamp(ScheduledDay)    AS scheduled_datetime,
  to_timestamp(AppointmentDay)  AS appointment_datetime,

  CAST(Age AS INT)              AS age,
  UPPER(TRIM(Gender))           AS gender,
  TRIM(Neighbourhood)           AS neighbourhood,

  CAST(Scholarship AS INT)      AS scholarship,
  CAST(Hipertension AS INT)     AS hypertension,
  CAST(Diabetes AS INT)         AS diabetes,
  CAST(Alcoholism AS INT)       AS alcoholism,
  CAST(Handcap AS INT)          AS handcap,
  CAST(SMS_received AS INT)     AS sms_received,

  CASE
    WHEN `No-show` = 'Yes' THEN 1
    WHEN `No-show` = 'No'  THEN 0
    ELSE NULL
  END AS no_show,

  CAST(to_timestamp(ScheduledDay)   AS DATE) AS scheduled_date,
  CAST(to_timestamp(AppointmentDay) AS DATE) AS appt_date,

  datediff(CAST(to_timestamp(AppointmentDay) AS DATE),
           CAST(to_timestamp(ScheduledDay)   AS DATE)) AS lead_time_days,

  dayofweek(CAST(to_timestamp(AppointmentDay) AS DATE)) AS dow_sun1,
  CASE WHEN dayofweek(CAST(to_timestamp(AppointmentDay) AS DATE)) IN (1,7) THEN 1 ELSE 0 END AS is_weekend,
  CASE WHEN CAST(Age AS INT) >= 60 THEN 1 ELSE 0 END AS is_senior

FROM bronze.appointments_raw
WHERE Age BETWEEN 0 AND 110;


## 2) Create `silver.patient_history_features` (No Leakage)

**Idea:** For each appointment, compute patient history using only *prior* appointments:
- `prev_appts`, `prev_noshows`, `prev_noshow_rate`, `last_appt_date`

This uses SQL window functions with `ROWS ... AND 1 PRECEDING` to prevent leakage.


In [0]:
%sql
CREATE OR REPLACE TABLE silver.patient_history_features
USING DELTA
AS
WITH base AS (
  SELECT
    patient_id,
    appointment_id,
    appointment_datetime,
    no_show
  FROM silver.appointments_clean
),
hist AS (
  SELECT
    patient_id,
    appointment_id,

    COUNT(*) OVER (
      PARTITION BY patient_id
      ORDER BY appointment_datetime
      ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS prev_appts,

    SUM(no_show) OVER (
      PARTITION BY patient_id
      ORDER BY appointment_datetime
      ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS prev_noshows,

    MAX(CAST(appointment_datetime AS DATE)) OVER (
      PARTITION BY patient_id
      ORDER BY appointment_datetime
      ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS last_appt_date
  FROM base
)
SELECT
  patient_id,
  appointment_id,
  COALESCE(prev_appts, 0) AS prev_appts,
  COALESCE(prev_noshows, 0) AS prev_noshows,
  CASE
    WHEN COALESCE(prev_appts, 0) = 0 THEN 0.0
    ELSE COALESCE(prev_noshows, 0) / COALESCE(prev_appts, 0)
  END AS prev_noshow_rate,
  last_appt_date
FROM hist;


## 3) Create `silver.model_features`

Join clean appointments with patient history and create a proxy risk segment based on lead time.


In [0]:
%sql
CREATE OR REPLACE TABLE silver.model_features
USING DELTA
AS
SELECT
  c.*,
  h.prev_appts,
  h.prev_noshows,
  h.prev_noshow_rate,
  h.last_appt_date,

  CASE
    WHEN c.lead_time_days >= 30 THEN 'LONG_LEAD'
    WHEN c.lead_time_days BETWEEN 7 AND 29 THEN 'MID_LEAD'
    WHEN c.lead_time_days BETWEEN 1 AND 6 THEN 'SHORT_LEAD'
    ELSE 'SAME_DAY_OR_NEG'
  END AS risk_segment_proxy

FROM silver.appointments_clean c
LEFT JOIN silver.patient_history_features h
  ON c.appointment_id = h.appointment_id;


## 4) Data Quality Gates (Silver)

Log checks to `dq.check_results`:
- No null keys
- Label validity (0/1)
- Lead time non-negative
- No duplicate appointment_id
- Row count comparison (Bronze vs Silver) with expected delta due to age filter


In [0]:
%sql
-- Null keys
INSERT INTO dq.check_results
SELECT
  current_timestamp(),
  'silver_no_null_ids',
  CASE WHEN COUNT(*)=0 THEN 'PASS' ELSE 'FAIL' END,
  COUNT(*),
  'appointment_id and patient_id must not be null'
FROM silver.appointments_clean
WHERE appointment_id IS NULL OR patient_id IS NULL;

-- Label valid
INSERT INTO dq.check_results
SELECT
  current_timestamp(),
  'silver_label_valid',
  CASE WHEN COUNT(*)=0 THEN 'PASS' ELSE 'FAIL' END,
  COUNT(*),
  'no_show must be 0/1'
FROM silver.appointments_clean
WHERE no_show NOT IN (0,1) OR no_show IS NULL;

-- Lead time non-negative
INSERT INTO dq.check_results
SELECT
  current_timestamp(),
  'silver_lead_time_non_negative',
  CASE WHEN COUNT(*)=0 THEN 'PASS' ELSE 'FAIL' END,
  COUNT(*),
  'lead_time_days must be >= 0'
FROM silver.appointments_clean
WHERE lead_time_days < 0;

-- Duplicates by appointment_id
INSERT INTO dq.check_results
SELECT
  current_timestamp(),
  'silver_no_duplicate_appointment_id',
  CASE WHEN COUNT(*)=0 THEN 'PASS' ELSE 'FAIL' END,
  COUNT(*),
  'duplicate appointment_id detected'
FROM (
  SELECT appointment_id
  FROM silver.appointments_clean
  GROUP BY appointment_id
  HAVING COUNT(*) > 1
) d;

-- Row count delta (bronze vs silver) — explain drops (age filter)
INSERT INTO dq.check_results
WITH bc AS (SELECT COUNT(*) c FROM bronze.appointments_raw),
     sc AS (SELECT COUNT(*) c FROM silver.appointments_clean)
SELECT
  current_timestamp(),
  'row_count_bronze_vs_silver',
  CASE WHEN (SELECT c FROM bc) = (SELECT c FROM sc) THEN 'PASS' ELSE 'FAIL' END,
  ABS((SELECT c FROM bc) - (SELECT c FROM sc)) AS failed_count,
  CONCAT('bronze=', (SELECT c FROM bc), '; silver=', (SELECT c FROM sc), '; delta expected due to age filter')
;


## 5) Validation / Profiling

Quick checks to confirm table sizes and feature sanity.


In [0]:
%sql
SELECT COUNT(*) AS clean_rows FROM silver.appointments_clean;
SELECT COUNT(*) AS feature_rows FROM silver.model_features;

SELECT *
FROM dq.check_results
ORDER BY run_time_stamp DESC
LIMIT 20;


## 6) Sanity Checks (optional)

Verify label distribution and bounds on history features.


In [0]:
%sql
SELECT no_show, COUNT(*) AS n
FROM silver.appointments_clean
GROUP BY no_show
ORDER BY no_show;

SELECT
  MAX(prev_appts) AS max_prev_appts,
  MAX(prev_noshow_rate) AS max_prev_noshow_rate
FROM silver.model_features;
